In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet').head(100000)

In [3]:
df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60

In [4]:
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

In [5]:
categorical = ['PULocationID', 'DOLocationID']
df[categorical] = df[categorical].astype(str)

In [6]:
train_dicts = df[categorical].to_dict(orient='records')

In [7]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [8]:
from sklearn.linear_model import LinearRegression
y_train = df['duration'].values
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [10]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Get the model's predictions
y_pred = lr.predict(X_train)

# Calculate the error (take the square root of the mean squared error)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"RMSE: {rmse:.2f} minutes")

RMSE: 7.08 minutes


In [11]:
import pickle

# Save the model (lr) and the vectorizer (dv) to a file
with open('lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [12]:
# Let's pretend a new taxi ride just happened: Zone 236 to Zone 239
new_ride = [{'PULocationID': '26', 'DOLocationID': '239'}]

# 1. Translate it using our saved dv
X_new = dv.transform(new_ride)

# 2. Ask the model to predict
prediction = lr.predict(X_new)

print(f"The model predicts this trip will take: {prediction[0]:.2f} minutes")

The model predicts this trip will take: 10.99 minutes
